In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch08-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "08-02-principal-component-analysis.py"

用 1,125 筆 ESOL 分子擬合四個描述符的 PCA，再將留下的蒽、菲和甘油投影到相同的座標。首次執行須下載 ESOL。要嘗試小練習，可在下方 %%writefile 程式格最後加入自己的程式，再重跑該格與最末格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import tempfile

import deepchem as dc
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rdkit import Chem
from rdkit.Chem import Crippen, Descriptors, Draw, rdMolDescriptors
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "cheminfo-datasets"))
cache.mkdir(parents=True, exist_ok=True)
_, datasets, _ = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], data_dir=str(cache), reload=False)
mols = [Chem.MolFromSmiles(str(s).strip()) for s in datasets[0].ids]
assert len(mols) == 1128 and all(m is not None for m in mols)
smiles = np.array([Chem.MolToSmiles(m) for m in mols])
names = ["MW", "clogP", "TPSA", "RotB"]

def describe(mol):
    return [Descriptors.MolWt(mol), Crippen.MolLogP(mol),
            rdMolDescriptors.CalcTPSA(mol), rdMolDescriptors.CalcNumRotatableBonds(mol)]

X = np.array([describe(m) for m in mols])
examples = {"Anthracene": "c1ccc2cc3ccccc3cc2c1", "Phenanthrene": "c1ccc2c(c1)ccc3ccccc32",
            "Glycerol": "OCC(O)CO"}
held = []
for structure in examples.values():
    matches = np.flatnonzero(smiles == Chem.MolToSmiles(Chem.MolFromSmiles(structure)))
    assert len(matches) == 1
    held.append(matches[0])
reference = np.ones(len(X), dtype=bool)
reference[held] = False
scaler = StandardScaler().fit(X[reference])
pca = PCA(svd_solver="full").fit(scaler.transform(X[reference]))
scores = pca.transform(scaler.transform(X))
raw = PCA(svd_solver="full").fit(X[reference])
np.set_printoptions(precision=4, suppress=True)
print("Reference rows:", reference.sum(), " / Standard deviations:", scaler.scale_)
print("Raw PC1 fraction and weights:", raw.explained_variance_ratio_[0], raw.components_[0])
print("Standardized variance fractions:", pca.explained_variance_ratio_)
for pc, weights in enumerate(pca.components_[:2], start=1):
    print(f"PC{pc} weights ({', '.join(names)}):", weights)
for name, i in zip(examples, held):
    print(name, X[i], "->", scores[i, :2])

fig = plt.figure(figsize=(9, 5.4), layout="constrained")
grid = fig.add_gridspec(3, 2, width_ratios=[2.1, 1])
ax = fig.add_subplot(grid[:, 0])
ax.scatter(*scores[reference, :2].T, s=13, c="#557782", linewidths=0, alpha=0.4)
ax.axhline(0, color="#adb5b9", lw=0.65, zorder=0)
ax.axvline(0, color="#adb5b9", lw=0.65, zorder=0)
fraction = pca.explained_variance_ratio_
ax.set(xlabel=f"PC1 ({fraction[0]:.1%})", ylabel=f"PC2 ({fraction[1]:.1%})",
       title="ESOL  |  1,125 reference molecules")
ax.set_aspect("equal", adjustable="box")
for label, i, color in [("A, B", held[0], "#a24222"), ("C", held[2], "#236931")]:
    x, y = scores[i, :2]
    ax.scatter(x, y, marker="D", s=54, color=color, edgecolor="white", zorder=3)
    ax.annotate(label, (x, y), xytext=(7, 7), textcoords="offset points",
                color=color, weight="bold")
for k, (name, i) in enumerate(zip(examples, held)):
    panel = fig.add_subplot(grid[k, 1])
    panel.imshow(Draw.MolToImage(mols[i], size=(340, 148)))
    panel.set_title(f"{'ABC'[k]}   {name}   ({scores[i, 0]:.2f}, {scores[i, 1]:.2f})", fontsize=10)
    panel.axis("off")
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(figure_dir / "08-02-principal-component-analysis.png", dpi=180)
plt.close(fig)


In [ ]:
from IPython.display import Image, display

subprocess.run([python, str(script)], check=True)
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "08-02-principal-component-analysis.png")))